In [4]:
import pandas as pd
import numpy as np

import tqdm
import glob

from pandas.tseries.offsets import MonthEnd

In [5]:
pd.set_option('display.max_rows', 500)
pd.set_option('display.max_columns', 500)
pd.set_option('display.width', 1000)

In [13]:
df_list = []
for file in tqdm.tqdm(glob.glob("gupshup_report_202501_202504/*.csv")):
    small_df = pd.read_csv(file, sep=",")
    df_list.append(small_df)

 21%|██        | 6/29 [00:10<00:38,  1.68s/it]/var/folders/3r/sqxygt_d4gs0bc8yhzxh19vw0000gp/T/ipykernel_36917/1319167959.py:3: DtypeWarning: Columns (34) have mixed types. Specify dtype option on import or set low_memory=False.
  small_df = pd.read_csv(file, sep=",")
 31%|███       | 9/29 [00:15<00:33,  1.68s/it]/var/folders/3r/sqxygt_d4gs0bc8yhzxh19vw0000gp/T/ipykernel_36917/1319167959.py:3: DtypeWarning: Columns (34) have mixed types. Specify dtype option on import or set low_memory=False.
  small_df = pd.read_csv(file, sep=",")
 41%|████▏     | 12/29 [00:20<00:28,  1.68s/it]/var/folders/3r/sqxygt_d4gs0bc8yhzxh19vw0000gp/T/ipykernel_36917/1319167959.py:3: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  small_df = pd.read_csv(file, sep=",")
 52%|█████▏    | 15/29 [00:25<00:24,  1.74s/it]/var/folders/3r/sqxygt_d4gs0bc8yhzxh19vw0000gp/T/ipykernel_36917/1319167959.py:3: DtypeWarning: Columns (32,33) have mixed types. Specify dtype op

In [14]:
small_df.head(2)

,PHONENO,SENDER,TRANSACTION ID,MESSAGE ID,ACCOUNT ID,TYPE,SENT,DELIVERED,READ,FAILED,STATUS,CAUSE,CHANNEL,TEMPLATE ID,NUMBER MESSAGES,DELIVERY CODE,DLT TEMPLATEID,PRINCIPAL ENTITYID,RETRY STATUS,CONVERSATION ID,CATEGORY,CATEGORY TYPE,EXTRA,PRICING CATEGORY,PROJECT ID,ORG ID,META ERROR CODE,META ERROR MESSAGE,TEMPLATE NAME,TEMPLATE LANGUAGE,REQUESTED,SUBMITTED,BUTTON CLICK TIMESTAMP,BUTTON NAME,LINK CLICK TIMESTAMP,CAMPAIGN ID
0,966541549598,966920022776,5393670881644052558,318141069329526609,2000235758,TEMPLATE,2025-03-20 21:47:33,2025-03-20 21:47:34,NaN,NaN,Delivered,Delivered,WHATSAPP,1349268,1,000,NaN,NaN,F,a54c8650866388c5cf1b10bd88b6c0e2,marketing,MC_PF,NaN,marketing,31566297,31254027,NaN,NaN,ar79179weekendoffer,ar,2025-03-20 21:45:01,2025-03-20 21:47:31,NaN,NaN,NaN,3a6b9b0a-6ce3-41e1-af7c-1878dbef31e5
1,966507976802,966920022776,5393670881644052558,318146286762489652,2000235758,TEMPLATE,2025-03-20 21:47:42,2025-03-20 21:48:06,2025-03-21 03:39:45,NaN,Delivered,Read,WHATSAPP,1349268,1,026,NaN,NaN,F,d203e035ee80df91cd3237ae01921f19,marketing,MC_PF,NaN,marketing,31566297,31254027,NaN,NaN,ar79179weekendoffer,ar,2025-03-20 21:45:01,2025-03-20 21:47:41,NaN,NaN,NaN,3a6b9b0a-6ce3-41e1-af7c-1878dbef31e5


In [15]:
small_df.head(1).T

,0
PHONENO,966541549598
SENDER,966920022776
TRANSACTION ID,5393670881644052558
MESSAGE ID,318141069329526609
ACCOUNT ID,2000235758
TYPE,TEMPLATE
SENT,2025-03-20 21:47:33
DELIVERED,2025-03-20 21:47:34
READ,NaN
FAILED,NaN


In [16]:
campaign_df = pd.concat(df_list)

In [17]:
campaign_df.shape, campaign_df.drop_duplicates().shape

((13035401, 36), (13035401, 36))

In [18]:
campaign_df["SENT"].isna().sum()

1697761

In [19]:
campaign_df["date"] = pd.to_datetime(campaign_df["SENT"], format="mixed").dt.date
campaign_df["_observ_end_dt"] = pd.to_datetime(campaign_df["date"]) + MonthEnd(1)
campaign_df["_observ_end_dt"] = campaign_df["_observ_end_dt"].dt.date

In [20]:
campaign_df["was_sent"] = (~campaign_df["SENT"].isnull()).astype(int)
campaign_df["was_delivered"] = (~campaign_df["DELIVERED"].isnull()).astype(int)
campaign_df["was_read"] = (~campaign_df["READ"].isnull()).astype(int)

In [21]:
campaign_df["was_sent"].value_counts()

was_sent
1    11337640
0     1697761
Name: count, dtype: int64

In [22]:
campaign_df["was_delivered"].value_counts()

was_delivered
1    10931991
0     2103410
Name: count, dtype: int64

In [23]:
campaign_df.groupby(["_observ_end_dt"]).agg(
    total_number_of_customers=("PHONENO", "count"),
    unique_number_of_customers=("PHONENO", "nunique"),
    sent=("was_sent", "sum"),
    delivered=("was_delivered", "sum"),
    read=("was_read", "sum"),
)

,total_number_of_customers,unique_number_of_customers,sent,delivered,read
_observ_end_dt,,,,,
2024-12-31,9914,8770,9914,9892,9218
2025-01-31,729454,584562,729454,705953,434553
2025-02-28,2185231,1013227,2185231,2129965,1323426
2025-03-31,4026958,1067720,4026958,3896825,2556220
2025-04-30,4365817,1337168,4365817,4169581,2454067
2025-05-31,20266,11854,20266,19775,12942


# join with Churn data

In [24]:
mdl_churn_predicted_df = pd.read_parquet("../data/07_model_output/churn/predict/mdl_churn_predicted")


In [25]:
prm_spine = pd.read_parquet("../data/03_primary/prm_spine")

In [26]:
mdl_churn_predicted_df[["plate_number", "mobile"]] = mdl_churn_predicted_df["_id"].str.rpartition("__")[[0,2]]
mdl_churn_predicted_df["n_cars"] = mdl_churn_predicted_df.groupby(["mobile"])["_id"].transform("nunique")
mdl_churn_predicted_df["churn_probability"] = np.where(mdl_churn_predicted_df["n_cars"] > 1, None, mdl_churn_predicted_df["churn_probability"])

In [27]:
mdl_churn_predicted_df[["target_churn_1", "target_churn_2"]].head()

,target_churn_1,target_churn_2
0,1,1.0
1,1,1.0
2,1,1.0
3,1,1.0
4,1,1.0


In [28]:
campaign_df["mobile"] = campaign_df["PHONENO"].astype(str)

In [29]:
selected_cols = ["_id", "_observ_end_dt", "plate_number", "mobile", "churn_probability", "is_active", "target_churn_1", "target_churn_2", "n_cars"]

In [30]:
mdl_churn_predicted_df["_observ_end_dt"].value_counts()

_observ_end_dt
2025-05-31    3121056
2025-04-30    3120712
2025-03-31    3082530
2025-02-28    3046560
2025-01-31    3011095
2024-12-31    2979087
Name: count, dtype: int64

In [31]:
campaign_df["_observ_end_dt"].value_counts()

_observ_end_dt
2025-04-30    4365817
2025-03-31    4026958
2025-02-28    2185231
2025-01-31     729454
2025-05-31      20266
2024-12-31       9914
Name: count, dtype: int64

In [32]:
joint_df = pd.merge(
    campaign_df,
    mdl_churn_predicted_df[selected_cols],
    on=["mobile", "_observ_end_dt"],
    how="outer"
)

In [33]:
joint_df = joint_df[~joint_df["churn_probability"].isnull()]

In [34]:
joint_df.shape

(17351429, 49)

In [ ]:
# joint_df["adjusted_min_churn_probability"] = joint_df.groupby(["mobile", "_observ_end_dt"])["churn_probability"].transform("min")
# joint_df["adjusted_max_churn_probability"] = joint_df.groupby(["mobile", "_observ_end_dt"])["churn_probability"].transform("max")
# joint_df["adjusted_min_target_churn_1"] = joint_df.groupby(["mobile", "_observ_end_dt"])["target_churn_1"].transform("min")
# joint_df["adjusted_min_target_churn_2"] = joint_df.groupby(["mobile", "_observ_end_dt"])["target_churn_2"].transform("min")

In [35]:
joint_df["churn_prediction"] = np.where(joint_df["churn_probability"] > 0.8, 1, 0)

In [36]:
joint_df.head()

,PHONENO,SENDER,TRANSACTION ID,MESSAGE ID,ACCOUNT ID,TYPE,SENT,DELIVERED,READ,FAILED,STATUS,CAUSE,CHANNEL,TEMPLATE ID,NUMBER MESSAGES,DELIVERY CODE,DLT TEMPLATEID,PRINCIPAL ENTITYID,RETRY STATUS,CONVERSATION ID,CATEGORY,CATEGORY TYPE,EXTRA,PRICING CATEGORY,PROJECT ID,ORG ID,META ERROR CODE,META ERROR MESSAGE,TEMPLATE NAME,TEMPLATE LANGUAGE,REQUESTED,SUBMITTED,BUTTON CLICK TIMESTAMP,BUTTON NAME,LINK CLICK TIMESTAMP,CAMPAIGN ID,date,_observ_end_dt,was_sent,was_delivered,was_read,mobile,_id,plate_number,churn_probability,is_active,target_churn_1,target_churn_2,n_cars,churn_prediction
65597,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2024-12-31,NaN,NaN,NaN,966500000002,8928AUB__966500000002,8928AUB,0.409999,0.0,1.0,1.0,1.0,0
65598,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2025-01-31,NaN,NaN,NaN,966500000002,8928AUB__966500000002,8928AUB,0.409999,0.0,1.0,1.0,1.0,0
65599,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2025-02-28,NaN,NaN,NaN,966500000002,8928AUB__966500000002,8928AUB,0.409999,0.0,1.0,1.0,1.0,0
65600,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2025-03-31,NaN,NaN,NaN,966500000002,8928AUB__966500000002,8928AUB,0.409999,0.0,1.0,1.0,1.0,0
65601,966500000002,9.669200e+11,5.414667e+18,331410487618348809,2.000236e+09,TEMPLATE,2025-04-18 21:04:23,2025-04-18 21:18:25,NaN,NaN,Delivered,Delivered,WHATSAPP,1431811,1.0,000,NaN,NaN,F,13a0b0030ab49bd1a7a1c0f632a3802f,marketing,MC,NaN,marketing,31566297.0,31254027.0,NaN,NaN,v2_lostgone_arabic,ar,2025-04-18 21:00:02,2025-04-18 21:04:21,NaN,NaN,NaN,b679d9c9-f266-4f87-a893-654b94c126aa,2025-04-18,2025-04-30,1.0,1.0,0.0,966500000002,8928AUB__966500000002,8928AUB,0.409999,0.0,1.0,1.0,1.0,0


In [ ]:
joint_df

In [ ]:
# "was_sent", "was_delivered",

In [ ]:
joint_df["was_delivered"] = np.where(joint_df["was_delivered"].isnull(), 0, joint_df["was_delivered"])
joint_df["was_read"] = np.where(joint_df["was_read"].isnull(), 0, joint_df["was_read"])

In [39]:
joint_df_agg = joint_df.groupby(["_observ_end_dt", "was_delivered", "was_read", "churn_prediction"]).agg(
    total_customers=("_id", 'count'),
    unique_customers=("_id", 'nunique'),
    churned=("target_churn_1", "mean")
)

In [40]:
joint_df_agg

total_customers  unique_customers   churned
_observ_end_dt was_delivered was_read churn_prediction                                             
2024-12-31     0.0           0.0      0                              15                13  0.933333
                                      1                               1                 1  1.000000
               1.0           0.0      0                             395               363  0.762025
                                      1                              83                78  0.963855
                             1.0      0                            5137              4635  0.775745
                                      1                             893               802  0.905935
2025-01-31     0.0           0.0      0                           17194             15772  0.943643
                                      1                             762               714  0.921260
               1.0           0.0      0                          180668            159712  0.886156
                                      1                           14712             11835  0.874048
                             1.0      0                          272189            223742  0.843370
                                      1                           27734             19616  0.881698
2025-02-28     0.0           0.0      0                           28250             14752  0.725947
                                      1                           10038              5522  0.940227
               1.0           0.0      0                          389159            198042  0.590473
                                      1                          160805             83057  0.920276
                             1.0      0                          630598            304466  0.553289
                                      1                          259021            126693  0.909254
2025-03-31     0.0           0.0      0                           72418             23232  0.848836
                                      1                           21992              7358  0.970080
               1.0           0.0      0                          707435            232040  0.742721
                                      1                          231922             78011  0.952359
                             1.0      0                         1316300            358371  0.708404
                                      1                          458365            129047  0.948997
2025-04-30     0.0           0.0      0                          137123             48901  0.967285
                                      1                           19158              8677  0.998382
               1.0           0.0      0                         1118569            400940  0.938040
                                      1                          194479             86856  0.998061
                             1.0      0                         1563163            498779  0.926129
                                      1                          286011            111257  0.997371
2025-05-31     0.0           0.0      0                             203               200  0.940887
                                      1                             146               146  1.000000
               1.0           0.0      0                            2513              1786  0.828492
                                      1                            1561              1261  0.999359
                             1.0      0                            4827              2601  0.848353
                                      1                            2469              1560  0.996760

In [ ]:
# How much are going to churn e.g P > 85%
# How much got a message
# How much actually churned?
# How much did not?


In [43]:
joint_df[joint_df["_observ_end_dt"] < pd.to_datetime("2025-05-01", format="%Y-%m-%d").date()].groupby(["was_delivered", "was_read", "churn_prediction"]).agg(
    total_customers=("_id", 'count'),
    unique_customers=("_id", 'nunique'),
    churned=("target_churn_1", "mean")
)

total_customers  unique_customers   churned
was_delivered was_read churn_prediction                                             
0.0           0.0      0                          255000             68496  0.905314
                       1                           51951             14071  0.974033
1.0           0.0      0                         2396226            607916  0.819989
                       1                          602001            149498  0.956641
              1.0      0                         3787387            791749  0.782230
                       1                         1032024            206916  0.950583

In [45]:
filter_before_may = joint_df["_observ_end_dt"] < pd.to_datetime("2025-05-01", format="%Y-%m-%d").date()
joint_df[filter_before_may].groupby(["churn_prediction"]).agg(
    total_customers=("_id", 'count'),
    unique_customers=("_id", 'nunique'),
    churned=("target_churn_1", "mean")
)

,total_customers,unique_customers,churned
churn_prediction,,,
0,12854751,1941160,0.833958
1,2437994,478570,0.944092


In [54]:
filter_before_may = joint_df["_observ_end_dt"] < pd.to_datetime("2025-04-01", format="%Y-%m-%d").date()
filter_delivered = joint_df["was_delivered"] > 0
joint_df[filter_before_may & filter_delivered].groupby(["churn_prediction", "was_read", "target_churn_1"]).agg(
    total_messages=("_id", 'count'),
    unique_customers=("_id", 'nunique'),
)

total_messages  unique_customers
churn_prediction was_read target_churn_1                                  
0                0.0      0.0                     362041            108458
                          1.0                     915616            314469
                 1.0      0.0                     709308            172679
                          1.0                    1514916            434486
1                0.0      0.0                      25725              9167
                          1.0                     381797            113575
                 1.0      0.0                      50248             15148
                          1.0                     695765            172473

In [55]:
filter_before_may = joint_df["_observ_end_dt"] < pd.to_datetime("2025-04-01", format="%Y-%m-%d").date()
filter_delivered = joint_df["was_delivered"] > 0
joint_df[filter_before_may & filter_delivered].groupby(["churn_prediction", "was_delivered", "target_churn_1"]).agg(
    total_messages=("_id", 'count'),
    unique_customers=("_id", 'nunique'),
)

total_messages  unique_customers
churn_prediction was_delivered target_churn_1                                  
0                1.0           0.0                    1071349            246109
                               1.0                    2430532            665189
1                1.0           0.0                      75973             22083
                               1.0                    1077562            246856

In [ ]:
filter_before_may = joint_df["_observ_end_dt"] < pd.to_datetime("2025-05-01", format="%Y-%m-%d").date()
filter_delivered = joint_df["was_delivered"] > 0
joint_df[filter_before_may & filter_delivered].groupby(["churn_prediction", "was_read",]).agg(
    total_messages=("_id", 'count'),
    unique_customers=("_id", 'nunique'),
    churned_perc=("target_churn_1", "mean"),
    churned_total=("target_churn_1", "sum"),
)

total_customers  unique_customers  churned_perc  churned_total
churn_prediction was_read                                                                
0                0.0               2396226            607916      0.819989      1964879.0
                 1.0               3787387            791749      0.782230      2962606.0
1                0.0                602001            149498      0.956641       575899.0
                 1.0               1032024            206916      0.950583       981024.0

In [ ]:
filter_before_may = joint_df["_observ_end_dt"] < pd.to_datetime("2025-05-01", format="%Y-%m-%d").date()
joint_df[filter_before_may].groupby(["churn_prediction", "was_delivered",]).agg(
    total_messages=("_id", 'count'),
    unique_customers=("_id", 'nunique'),
    churned_perc=("target_churn_1", "mean"),
    churned_total=("target_churn_1", "sum"),
)

total_customers  unique_customers  churned_perc  churned_total
churn_prediction was_delivered                                                                
0                0.0                     255000             68496      0.905314       230855.0
                 1.0                    6183613           1153507      0.796862      4927485.0
1                0.0                      51951             14071      0.974033        50602.0
                 1.0                    1634025            294150      0.952815      1556923.0

In [11]:
ago_list = [
"966581419661",
"966563556470",
"966533984159",
"966554127441",
"966554993775",
"966530355656",
"966546191757",
"966559088020",
"966566044673",
"966532145810",
"966558018222",
"966595603080",
"966507710775",
"966509374131",
"966538882023",
"966533069297",
"966597071217",
"966581409184",
"966561499700",
"966546479213",
"966502585422",
"966559956656",
"966562521309",
"966553108020",
"966544481996",
"966503473246",
"966541799970",
"966502016665",
"966500505101",
"966502324402",
"966542826775",
"966594391426",
"966535138917",
"966539504364",
"966504343682",
"966504223730",
"966591726861",
"966500635398",
"966501077545",
"966531558074",
"966533404619",
"966555232712",
"966544908143",
"966548701404",
"966554443844",
"966542522117",
"966545849023",
"966504726043",
"966566796187",
"966565174470",
"966506160189",
"966556251112",
"966561956428",
"966505922457",
"966555711341",
"966544940754",
"966542684674",
"966500369523",
"966551404504",
"966568880300",
"966537077741",
"966504632321",
"966531699938",
"966540853457",
"966500403250",
"966566040033",
"966594141694",
"966581988199",
"966551368607",
"966505777041",
"966509998241",
"966554214274",
"966553274062",
"966564257868",
"966532777927",
"966572926585",
"966544008750",
"966561000485",
"966537317436",
"966502591562",
"966557308804",
"966537994899",
"966533302667",
"966593500757",
"966534221268",
"966555151417",
"966583606601",
"966594300693",
"966563127258",
"966503676333",
"966505346460",
"966596657101",
"966559127473",
"966508348041",
"966557889228",
"966572235854",
"966542075220",
"966566907003",
"966502860534",
"966500521476",
]

In [12]:
filter_mobiles = mdl_churn_predicted_df["mobile"].isin(ago_list)
filter_may = pd.to_datetime(mdl_churn_predicted_df["_observ_end_dt"], format="%Y-%m-%d") == pd.to_datetime("2025-05-31", format="%Y-%m-%d")
filters = filter_mobiles & filter_may

mdl_churn_predicted_df[filters][["mobile", "_observ_end_dt", "is_new_joiner"]]

,mobile,_observ_end_dt,is_new_joiner
1037382,966541799970,2025-05-31,0
2624218,966566040033,2025-05-31,0
3910820,966545849023,2025-05-31,0
4007445,966532777927,2025-05-31,0
6385888,966546479213,2025-05-31,1
6541625,966554127441,2025-05-31,0
6816974,966546479213,2025-05-31,0
6880112,966566040033,2025-05-31,0
8202283,966502324402,2025-05-31,0
8271865,966542826775,2025-05-31,0


In [10]:
mdl_churn_predicted_df.head()

,_id,_observ_end_dt,month_total_sales,month_net_sales,month_total_profit,month_total_cost,month_total_amount_discounts,month_total_qty_discounts,month_distinct_skus_sold,month_distinct_branches,month_distinct_product_categories,month_distinct_transactions,month_avg_order,month_avg_skus_per_order,month_net_sales_category_ac,month_total_profit_category_ac,month_total_cost_category_ac,month_total_amount_discounts_category_ac,month_total_qty_discounts_category_ac,month_net_sales_category_additives,month_total_profit_category_additives,month_total_cost_category_additives,month_total_amount_discounts_category_additives,month_total_qty_discounts_category_additives,month_net_sales_category_air_filter,month_total_profit_category_air_filter,month_total_cost_category_air_filter,month_total_amount_discounts_category_air_filter,month_total_qty_discounts_category_air_filter,month_net_sales_category_batteries,month_total_profit_category_batteries,month_total_cost_category_batteries,month_total_amount_discounts_category_batteries,month_total_qty_discounts_category_batteries,month_net_sales_category_brake,month_total_profit_category_brake,month_total_cost_category_brake,month_total_amount_discounts_category_brake,month_total_qty_discounts_category_brake,month_net_sales_category_coolant,month_total_profit_category_coolant,month_total_cost_category_coolant,month_total_amount_discounts_category_coolant,month_total_qty_discounts_category_coolant,month_net_sales_category_engine,month_total_profit_category_engine,month_total_cost_category_engine,month_total_amount_discounts_category_engine,month_total_qty_discounts_category_engine,month_net_sales_category_fuel,month_total_profit_category_fuel,month_total_cost_category_fuel,month_total_amount_discounts_category_fuel,month_total_qty_discounts_category_fuel,month_net_sales_category_gear,month_total_profit_category_gear,month_total_cost_category_gear,month_total_amount_discounts_category_gear,month_total_qty_discounts_category_gear,month_net_sales_category_oil,month_total_profit_category_oil,month_total_cost_category_oil,month_total_amount_discounts_category_oil,month_total_qty_discounts_category_oil,month_net_sales_category_oil_filter,month_total_profit_category_oil_filter,month_total_cost_category_oil_filter,month_total_amount_discounts_category_oil_filter,month_total_qty_discounts_category_oil_filter,month_net_sales_category_oil_synthetic,month_total_profit_category_oil_synthetic,month_total_cost_category_oil_synthetic,month_total_amount_discounts_category_oil_synthetic,month_total_qty_discounts_category_oil_synthetic,month_net_sales_category_others,month_total_profit_category_others,month_total_cost_category_others,month_total_amount_discounts_category_others,month_total_qty_discounts_category_others,month_net_sales_category_plug,month_total_profit_category_plug,month_total_cost_category_plug,month_total_amount_discounts_category_plug,month_total_qty_discounts_category_plug,month_net_sales_category_transmission,month_total_profit_category_transmission,month_total_cost_category_transmission,month_total_amount_discounts_category_transmission,month_total_qty_discounts_category_transmission,month_net_sales_category_tyres,month_total_profit_category_tyres,month_total_cost_category_tyres,month_total_amount_discounts_category_tyres,month_total_qty_discounts_category_tyres,most_trx_branch_branch_id,most_trx_branch_n_places_within_0300m__community,most_trx_branch_n_places_within_0300m__educational_establishment,most_trx_branch_n_places_within_0300m__entertainment,most_trx_branch_n_places_within_0300m__facilities,most_trx_branch_n_places_within_0300m__hospitals,most_trx_branch_n_places_within_0300m__marketplaces,most_trx_branch_n_places_within_0300m__payments,most_trx_branch_n_places_within_0300m__restaurants,most_trx_branch_n_places_within_0300m__transportation,most_trx_branch_n_places_within_0300m__All,most_trx_branch_n_places_within_0500m__community,most_trx_branch_n_places_within_0500m__educational_establ